In [1]:
import ipynbname
from pathlib import Path
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

from datasets import load_dataset

dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

/home/prga/Documents/uni/nlp/repos/msc-nlp-2026/project_notebooks


/home/prga/Documents/uni/nlp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
prime_train = df_train[(df_train["lang"].isin(['ar', 'ko', 'te']))]
prime_val   = df_val[(df_val["lang"].isin(['ar', 'ko', 'te']))]

# W 37

Assignment text:
> Convert the character-level answer offsets into BIO labels over context tokens. Add automatic checks for at least the following cases: an answer at character 0, a multi-token answer, punctuation adjacent to an answer and an unanswerable example. Document how subword pieces are handled if applicable. Implement one question-conditioned sequence labeller for the group: the representation of the question must influence the predicted label for every context token. Compare it with a simple span baseline. An empty-output baseline is sufficient. If you use lexical overlap, select context tokens using only overlap with the question (optionally after fixed preprocessing or translation), convert the best contiguous run to a span and never use gold answer text or offsets. The correct output for an unanswerable question is an empty span. Evaluate and analyse the models according to Section 

In [10]:
import nltk
import pandas as pd

In [11]:
def elaborate_df(df):
    df_plus = df
    df_plus['q_tokens'] = df_plus['question'].apply(lambda x: nltk.tokenize.word_tokenize(x))
    df_plus['c_tokens'] = df_plus['context'].apply(lambda x: nltk.tokenize.word_tokenize(x))
    df_plus['q_tokens_len'] = df_plus['q_tokens'].apply(lambda x: len(x))
    df_plus['c_tokens_len'] = df_plus['c_tokens'].apply(lambda x: len(x))
    return df_plus

In [17]:
#train_set = elaborate_df(prime_train)
#val_set = elaborate_df(prime_val)


Now begin

In [18]:
import random
import re

import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

random.seed(42)
np.random.seed(42)

In [20]:
nltk.download('averaged_perceptron_tagger_eng')



/home/prga/Documents/uni/nlp/.venv/lib/python3.13/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/prga/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/prga/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [23]:
prime_train['q_pos_tags'] = prime_train['q_tokens'].apply(lambda x: nltk.pos_tag(x))
prime_train['c_pos_tags'] = prime_train['c_tokens'].apply(lambda x: nltk.pos_tag(x))
prime_val['q_pos_tags'] = prime_val['q_tokens'].apply(lambda x: nltk.pos_tag(x))
prime_val['c_pos_tags'] = prime_val['c_tokens'].apply(lambda x: nltk.pos_tag(x))

/tmp/ipykernel_44013/357743687.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prime_train['q_pos_tags'] = prime_train['q_tokens'].apply(lambda x: nltk.pos_tag(x))
/tmp/ipykernel_44013/357743687.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prime_train['c_pos_tags'] = prime_train['c_tokens'].apply(lambda x: nltk.pos_tag(x))
/tmp/ipykernel_44013/357743687.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value

In [25]:
prime_train[(prime_train['lang'] == 'ar')].head(5)

,question,context,lang,answerable,answer_start,answer,answer_inlang,q_tokens,c_tokens,q_tokens_len,c_tokens_len,q_pos_tags,c_pos_tags
11213,متى تدخلت روسيا في الحرب الأهلية السورية؟,The Russian military intervention in the Syria...,ar,True,67,September 2015,None,"[متى, تدخلت, روسيا, في, الحرب, الأهلية, السورية؟]","[The, Russian, military, intervention, in, the...",7,147,"[(متى, JJ), (تدخلت, NNP), (روسيا, NNP), (في, N...","[(The, DT), (Russian, JJ), (military, JJ), (in..."
11214,متى حصلت هنغاريا على استقلالها من النمسا ؟,"By 1918, the economic situation had deteriorat...",ar,True,454,October 1918,None,"[متى, حصلت, هنغاريا, على, استقلالها, من, النمس...","[By, 1918, ,, the, economic, situation, had, d...",8,87,"[(متى, JJ), (حصلت, NNP), (هنغاريا, NNP), (على,...","[(By, IN), (1918, CD), (,, ,), (the, DT), (eco..."
11215,متى تحالفت فرنسا و بريطانيا العظمى ضد ألمانيا ...,France and Britain declared war on Germany whe...,ar,True,81,1939,None,"[متى, تحالفت, فرنسا, و, بريطانيا, العظمى, ضد, ...","[France, and, Britain, declared, war, on, Germ...",10,48,"[(متى, JJ), (تحالفت, NNP), (فرنسا, NNP), (و, N...","[(France, NNP), (and, CC), (Britain, NNP), (de..."
11216,كم عدد ضحايا أول إعتداء إسرائيلي على مدينة غزة ؟,The 2014 Israel–Gaza conflict also known as Op...,ar,True,607,death of thousands of people,None,"[كم, عدد, ضحايا, أول, إعتداء, إسرائيلي, على, م...","[The, 2014, Israel, –, Gaza, conflict, also, k...",10,125,"[(كم, JJ), (عدد, NNP), (ضحايا, NNP), (أول, NNP...","[(The, DT), (2014, CD), (Israel, NNP), (–, NNP..."
11217,هل سلسلة هاري بوتر مخالفة لقوانين المسيحية ؟,"Religious debates over the ""Harry Potter"" seri...",ar,False,-1,no,None,"[هل, سلسلة, هاري, بوتر, مخالفة, لقوانين, المسي...","[Religious, debates, over, the, ``, Harry, Pot...",8,188,"[(هل, JJ), (سلسلة, NNP), (هاري, NNP), (بوتر, N...","[(Religious, JJ), (debates, NNS), (over, IN), ..."


In [ ]:
def valid_bio_transition(previous, current):
    curr_tag = current.tag
    if not current_tag.startswith("I-"):
        return True
    if previous is None:
        return False
    entity_type = curr_tag[2:]
    return previous.tag in {f"B-{entity_type}", f"I-{entity_type}"}

def repair_bio(sequence):
    repaired = []
    for label in sequence:
        if valid_bio_transition(repaired[-1] if repaired else None, label):
            repaired.append(label)
        else:
            repaired.append(

In [31]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets


def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != answer_end:
        raise ValueError("The answer span does not align with token boundaries")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

In [32]:
context = "Ada Lovelace wrote the first algorithm."
answer_text = "Ada Lovelace"
answer_start = context.index(answer_text)

tokens, offsets, labels = character_span_to_bio(
    context, answer_start, answer_text
)
round_trip_start, round_trip_text = bio_to_character_span(
    context, offsets, labels
)

assert answer_start == 0
assert round_trip_start == answer_start
assert round_trip_text == answer_text
assert labels[:2] == ["B-ANS", "I-ANS"]

pd.DataFrame({"token": tokens, "offset": offsets, "label": labels})

,token,offset,label
0,Ada,"(0, 3)",B-ANS
1,Lovelace,"(4, 12)",I-ANS
2,wrote,"(13, 18)",O
3,the,"(19, 22)",O
4,first,"(23, 28)",O
5,algorithm,"(29, 38)",O
6,.,"(38, 39)",O


In [33]:
unanswerable_tokens, unanswerable_offsets, unanswerable_labels = character_span_to_bio(
    context, None, ""
)
assert set(unanswerable_labels) == {"O"}

second_context = "The event was on 1 July 2000."
second_answer = "1 July 2000"
second_start = second_context.index(second_answer)
second_tokens, second_offsets, second_labels = character_span_to_bio(
    second_context, second_start, second_answer
)
assert bio_to_character_span(
    second_context, second_offsets, second_labels
) == (second_start, second_answer)

In [35]:
print("hello")

hello


In [139]:
def answer_offset(context, tokens, answer_start, answer_length):
    stop_length = len(context[:answer_start - 1])
    curr_length = 0
    index = 0
    idx = []
    while curr_length <= stop_length:
        curr_length += len(tokens[index]) + 1
        index += 1
    
    #if index == len(tokens):
    #    idx.append(index)
    #    return idx
    
    while curr_length < answer_start + answer_length and index < len(tokens):
        idx.append(index)
        curr_length += len(tokens[index])
        index += 1
    return idx

In [110]:
a = prime_train.head(1)
answer_offset(a['context'].item(), a['c_tokens'].item(), a['answer_start'].item(), len(a['answer'].item()))

[3]

In [111]:
print(a['answer'].item())
print(a['answer_start'].item())
#print(a['c_tokens'].item())

France
21


In [141]:
def bio_tokenize(tokens, idx):
    bio_tokens = ["O"] * len(tokens)
    #print(f"{len(tokens)}-{idx}")
    if len(idx) == 0:
        return []
    for index in idx:
        bio_tokens[index] = "I-ANS"
    bio_tokens[idx[0]] = "B-ANS"
    return bio_tokens

In [129]:
idx = answer_offset(a['context'].item(), a['c_tokens'].item(), a['answer_start'].item(), len(a['answer'].item()))
bio_tokens = bio_tokenize(a['c_tokens'].item(), idx)
print(bio_tokens)
print(a['c_tokens'])
#print(a['bio_tokens'])


['O', 'O', 'O', 'B-ANS', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
4792    [The, conflict, between, France, and, Spain, c...
Name: c_tokens, dtype: object


In [142]:
prime_train['bio_tokens'] = prime_train.apply(lambda x: bio_tokenize(x['c_tokens'], answer_offset(x['context'], x['c_tokens'], x['answer_start'], len(x['answer']))), axis=1)
prime_train.head(2)
#prime_train.head(5).apply(lambda x: print(x['c_tokens']), axis=1)

/tmp/ipykernel_44013/3131946274.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prime_train['bio_tokens'] = prime_train.apply(lambda x: bio_tokenize(x['c_tokens'], answer_offset(x['context'], x['c_tokens'], x['answer_start'], len(x['answer']))), axis=1)


,question,context,lang,answerable,answer_start,answer,answer_inlang,q_tokens,c_tokens,q_tokens_len,c_tokens_len,q_pos_tags,c_pos_tags,bio_tokens
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,"[30년, 전쟁의, 승자는, 누구인가, ?]","[The, conflict, between, France, and, Spain, c...",5,120,"[(30년, CD), (전쟁의, JJ), (승자는, NN), (누구인가, NN), ...","[(The, DT), (conflict, NN), (between, IN), (Fr...","[O, O, O, B-ANS, O, O, O, O, O, O, O, O, O, O,..."
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None,"[엑스선은, 누가, 발견하였는가, ?]","[X-rays, make, up, X-radiation, ,, a, form, of...",4,174,"[(엑스선은, JJ), (누가, NNP), (발견하였는가, NN), (?, .)]","[(X-rays, NNS), (make, VBP), (up, RP), (X-radi...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [143]:
prime_train.head(10).apply(lambda x: print(x['bio_tokens']), axis=1)

['O', 'O', 'O', 'B-ANS', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O',

4792    None
4793    None
4794    None
4795    None
4796    None
4797    None
4798    None
4799    None
4800    None
4801    None
dtype: object